# 01 Daten laden und prüfen

Ziel: Überblick über alle sechs Tabellen des Maven-Toys-Datensatzes (Aufbau, Datentypen, fehlende Werte, Auffälligkeiten). In diesem Notebook werden keine Daten verändert. Die Bereinigung folgt in Notebook 02.

## Daten laden

In [ ]:
import pandas as pd

## Daten laden
products = pd.read_csv("../data/raw/products.csv")
stores = pd.read_csv("../data/raw/stores.csv")
sales = pd.read_csv("../data/raw/sales.csv")
inventory = pd.read_csv("../data/raw/inventory.csv")
calendar = pd.read_csv("../data/raw/calendar.csv")
data_dictionary = pd.read_csv("../data/raw/data_dictionary.csv")

## products
35 Produkte mit Kategorie, Einkaufspreis (Cost) und Verkaufspreis (Price).

**Auffälligkeiten:**
- `Product_Cost` und `Product_Price` sind Text mit Dollarzeichen (z. B. "$9.99") und müssen in Zahlen umgewandelt werden.
- Keine fehlenden Werte.

In [22]:
## products
products.head()

,Product_ID,Product_Name,Product_Category,Product_Cost,Product_Price
0,1,Action Figure,Toys,$9.99,$15.99
1,2,Animal Figures,Toys,$9.99,$12.99
2,3,Barrel O' Slime,Art & Crafts,$1.99,$3.99
3,4,Chutes & Ladders,Games,$9.99,$12.99
4,5,Classic Dominoes,Games,$7.99,$9.99


In [23]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Product_ID        35 non-null     int64
 1   Product_Name      35 non-null     str  
 2   Product_Category  35 non-null     str  
 3   Product_Cost      35 non-null     str  
 4   Product_Price     35 non-null     str  
dtypes: int64(1), str(4)
memory usage: 1.5 KB


## stores
50 Filialen mit Stadt, Standorttyp (Residential, Commercial, Downtown) und Eröffnungsdatum.

**Auffälligkeiten:**
- `Store_Open_Date` ist Text und muss in ein Datum umgewandelt werden.
- Keine fehlenden Werte.

In [24]:
## stores
stores.head()

,Store_ID,Store_Name,Store_City,Store_Location,Store_Open_Date
0,1,Maven Toys Guadalajara 1,Guadalajara,Residential,1992-09-18
1,2,Maven Toys Monterrey 1,Monterrey,Residential,1995-04-27
2,3,Maven Toys Guadalajara 2,Guadalajara,Commercial,1999-12-27
3,4,Maven Toys Saltillo 1,Saltillo,Downtown,2000-01-01
4,5,Maven Toys La Paz 1,La Paz,Downtown,2001-05-31


In [25]:
stores.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Store_ID         50 non-null     int64
 1   Store_Name       50 non-null     str  
 2   Store_City       50 non-null     str  
 3   Store_Location   50 non-null     str  
 4   Store_Open_Date  50 non-null     str  
dtypes: int64(1), str(4)
memory usage: 2.1 KB


## sales
Einzelne Verkäufe (829.262 Zeilen) mit Datum, Filiale, Produkt und verkauften Stück (`Units`).

**Auffälligkeiten:**
- `Date` ist Text und muss in ein Datum umgewandelt werden.
- Keine Umsatz- oder Gewinnspalte. Der Gewinn wird später über den Join mit `products` berechnet.
- Keine fehlenden Werte.
- Weitere Prüfungen (Duplikate, Zeitraum, IDs): siehe unten.

In [26]:
## sales
sales.head()   

,Sale_ID,Date,Store_ID,Product_ID,Units
0,1,2022-01-01,24,4,1
1,2,2022-01-01,28,1,1
2,3,2022-01-01,6,8,1
3,4,2022-01-01,48,7,1
4,5,2022-01-01,44,18,1


In [27]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 829262 entries, 0 to 829261
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   Sale_ID     829262 non-null  int64
 1   Date        829262 non-null  str  
 2   Store_ID    829262 non-null  int64
 3   Product_ID  829262 non-null  int64
 4   Units       829262 non-null  int64
dtypes: int64(4), str(1)
memory usage: 31.6 MB


In [33]:
print("Duplikate:", sales.duplicated().sum())
print("Sale_ID eindeutig:", sales["Sale_ID"].is_unique)
print("Erstes Datum:", sales["Date"].min())
print("Letztes Datum:", sales["Date"].max())
print("Product_IDs ohne Eintrag in products:", set(sales["Product_ID"]) - set(products["Product_ID"]))
print("Store_IDs ohne Eintrag in stores:", set(sales["Store_ID"]) - set(stores["Store_ID"]))

Duplikate: 0
Sale_ID eindeutig: True
Erstes Datum: 2022-01-01
Letztes Datum: 2023-09-30
Product_IDs ohne Eintrag in products: set()
Store_IDs ohne Eintrag in stores: set()


### Prüfung sales: Inhalt
- Duplikate: 0, `Sale_ID` ist eindeutig.
- Zeitraum: 2022-01-01 bis 2023-09-30 (638 Tage, passt zu `calendar`).
- Alle `Product_ID` und `Store_ID` sind in den Stammdaten vorhanden, der Join funktioniert ohne Verlust.

## inventory
Lagerbestand (`Stock_On_Hand`) je Filiale und Produkt, 1.593 Zeilen.

**Auffälligkeiten:**
- Kein Datum: Die Tabelle ist eine Momentaufnahme. Die Kapitalbindung lässt sich daher nur für einen Stichtag berechnen (`Stock_On_Hand` mal `Product_Cost`).
- 1.593 Zeilen statt 1.750 (50 Filialen mal 35 Produkte): Nicht jede Filiale führt jedes Produkt.
- Keine fehlenden Werte.

In [28]:
## inventory
inventory.head()

,Store_ID,Product_ID,Stock_On_Hand
0,1,1,27
1,1,2,0
2,1,3,32
3,1,4,6
4,1,5,0


In [29]:
inventory.info()

<class 'pandas.DataFrame'>
RangeIndex: 1593 entries, 0 to 1592
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Store_ID       1593 non-null   int64
 1   Product_ID     1593 non-null   int64
 2   Stock_On_Hand  1593 non-null   int64
dtypes: int64(3)
memory usage: 37.5 KB


## calendar
Tabelle mit 638 Tagen, nur die Spalte `Date`.

**Auffälligkeiten:**
- `Date` ist Text im Format Monat/Tag/Jahr (z. B. "1/1/2022"). In `sales` und `stores` steht das Datum als Jahr-Monat-Tag ("2022-01-01"). Beim Umwandeln muss das Format angegeben werden.

In [30]:
## calendar
calendar.head()

,Date
0,1/1/2022
1,1/2/2022
2,1/3/2022
3,1/4/2022
4,1/5/2022


In [31]:
calendar.info()

<class 'pandas.DataFrame'>
RangeIndex: 638 entries, 0 to 637
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Date    638 non-null    str  
dtypes: str(1)
memory usage: 5.1 KB


## data_dictionary
Beschreibung der Felder aller Tabellen (19 Zeilen). Dient als Nachschlagewerk, wird nicht analysiert.

In [32]:
## data dictionary
data_dictionary

,Table,Field,Description
0,Products,Product_ID,Product ID
1,Products,Product_Name,Product name
2,Products,Product_Category,Product Category
3,Products,Product_Cost,Product cost ($USD)
4,Products,Product_Price,Product retail price ($USD)
5,Inventory,Store_ID,Store ID
6,Inventory,Product_ID,Product ID
7,Inventory,Stock_On_Hand,Stock quantity of the product in the store (in...
8,Stores,Store_ID,Store ID
9,Stores,Store_Name,Store name


## Zusammenfassung der Auffälligkeiten

Aufgabenliste für Notebook 02:
1. `products`: `Product_Cost` und `Product_Price` von Text in Zahlen umwandeln.
2. `stores`, `sales`, `calendar`: Datumsspalten in echte Datumswerte umwandeln (`calendar` hat ein anderes Format).
3. `sales` und `products` verbinden und die Gewinnspalte berechnen.
4. `inventory`: Kapitalbindung als `Stock_On_Hand` mal `Product_Cost` berechnen.
5. Bereinigte Tabellen in `data/processed` speichern.